# EDA und Datenvorbereitung

Ziel dieses Notebooks ist es, die Data-Collection-Outputs zu verstehen, erste Qualitätschecks durchzuführen und die Datasets für die Analyse vorzubereiten

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

In [ ]:
DATA_DIR = Path("../../data")
INTERIM_DIR = DATA_DIR / "interim"

In [ ]:
COSTS_FILE = INTERIM_DIR / "gesundheitskosten_2011_2026.csv"

df_costs = pd.read_csv(COSTS_FILE)

print(pd.DataFrame({
        "rows": [len(df_costs)],
        "columns": [df_costs.shape[1]],
        "duplicate_rows": [df_costs.duplicated().sum()],
        "missing_cells": [df_costs.isna().sum().sum()],
}))
empty_columns = df_costs.columns[df_costs.isna().all()].tolist()
print(empty_columns)

Die Gesundheitskosten enthalten 44’232 fehlende Werte. Diese entstehen  durch sechs optionale Metadatenspalten die leer sind. Die analyse-relevanten Spalten wie Jahr, Kanton, Alter, Beobachtungswert, Multiplikator und Status enthalten keine fehlenden Werte.

Es sind ebenfalls viele Spalten vorhanden, welche für die Analyse später nicht relevant sind.

In [ ]:
cost_columns = [
    "TIME_PERIOD",
    "CANTON",
    "Swiss cantons",
    "AGE",
    "Age groups",
    "OBS_VALUE",
    "MULT",
    "OBS_STATUS",
    "Code list for Observation Status",
]

df_costs_processed = df_costs[cost_columns].copy()
df_costs_processed.head(30)

In [ ]:
pd.DataFrame({
        "year_min": [df_costs_processed["TIME_PERIOD"].min()],
        "year_max": [df_costs_processed["TIME_PERIOD"].max()],
        "n_years": [df_costs_processed["TIME_PERIOD"].nunique()],
        "n_cantons": [df_costs_processed["Swiss cantons"].nunique()],
        "n_age_groups": [df_costs_processed["Age groups"].nunique()],
        "n_status": [df_costs_processed["OBS_STATUS"].nunique()],
        "n_multipliers": [df_costs_processed["MULT"].nunique()],
})

In [ ]:
print(f'Status: \n {df_costs_processed["OBS_STATUS"].unique()}')
print(f'Kantone: \n {df_costs_processed["Swiss cantons"].unique()}')

In [ ]:
pd.crosstab(
    df_costs_processed["TIME_PERIOD"],
    df_costs_processed["OBS_STATUS"]
)

Für den Status haben 3 verschiedene Werte:

- A: Normaler Wert
- E: Geschätzter Wert
- P: Provisorischer Wert

Das Jahr 2023 beinhaltet nur provisorische Werte für alle Zeilen.
Das Jahr 2024 beinhaltet insgesamt nur einen geschätzten Wert.

Die Spalte Kantone beinhaltet alle 26 Kantone und das Total für die Schweiz

Um den absolut Wert für die Kosten zu erhalten müssen wird eine zusätzliche Spalte einfügen

In [ ]:
df_costs_processed["costs_chf"] = df_costs["OBS_VALUE"] * (10 ** df_costs["MULT"])
df_costs_processed.head()

Die Spalten sollten für einfacheres handling umbenennt werden.

In [ ]:
df_costs_processed = df_costs_processed.rename(columns={
    "TIME_PERIOD": "year",
    "CANTON": "canton_label",
    "Swiss cantons": "cantons",
    "AGE": "age_label",
    "Age groups": "age",
    "OBS_VALUE": "cost_raw",
    "MULT": "mult",
    "OBS_STATUS": "cost_status_label",
    "Code list for Observation Status": "cost_status",
})
df_costs_processed.head()

In [ ]:
df_costs_total_ch = df_costs_processed[
    (df_costs_processed["cantons"] == "Total") &
    (df_costs_processed["age"] == "Total")
].copy()

px.line(
    df_costs_total_ch,
    x="year",
    y="costs_chf",
    markers=True,
    title="Gesundheitskosten Schweiz, Total",
    labels={"year": "Jahr", "costs_chf": "Kosten in CHF"},
)

In [ ]:
df_costs_canton_total = df_costs_processed[
    (df_costs_processed["age"] == "Total") &
    (df_costs_processed["cantons"] != "Total")
].copy()

latest_canton_year = int(df_costs_canton_total["year"].max())

df_costs_canton_latest = (
    df_costs_canton_total[df_costs_canton_total["year"] == latest_canton_year]
    .sort_values("costs_chf", ascending=False)
)

px.bar(
    df_costs_canton_latest.head(10),
    x="cantons",
    y="costs_chf",
    title=f"Top 10 Kantone nach Gesundheitskosten {latest_canton_year}",
)

Im Plot für die Top 10 Kantone ist zu sehen, dass die Bevölkerungsgrösse eine Rolle spielt.

Für die spätere Analyse könnten die pro Kopf Kosten einen besseren Einblick geben

In [ ]:
df_costs_age_ch = df_costs_processed[
    (df_costs_processed["cantons"] == "Total") &
    (df_costs_processed["age"] != "Total") &
    (df_costs_processed["year"] == 2023)
].copy()

df_costs_age_ch["cost_share"] = (
    df_costs_age_ch["costs_chf"] / df_costs_age_ch["costs_chf"].sum()
)

px.bar(
    df_costs_age_ch,
    x="age",
    y="cost_share",
    title="Kostenanteil nach Altersgruppe Schweiz 2023",
)

In [ ]:
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

df_costs_processed.to_csv(PROCESSED_DIR / "gesundheitskosten.csv", index=False)

# 2. Bevölkerung

In [ ]:
POPULATION_FILE = INTERIM_DIR / "bevoelkerung_2011_2026.csv"

df_population = pd.read_csv(POPULATION_FILE)
pd.DataFrame({
        "rows": [len(df_population)],
        "columns": [df_population.shape[1]],
        "duplicate_rows": [df_population.duplicated().sum()],
        "missing_cells": [df_population.isna().sum().sum()],
})

In [ ]:
df_population.head()

In [ ]:
print(df_population["Staatsangehörigkeit (Kategorie)"].unique())
print(df_population["Geschlecht"].unique())

Die Spalten Staatsangehörige und Geschlecht enthalten für alle Zeilen die selben Werte. Für die weitere Analyse sind diese nicht von Bedeutung.

In [ ]:
df_population_processed = df_population.drop(columns=["Staatsangehörigkeit (Kategorie)", "Geschlecht"])
print(pd.DataFrame({
        "rows": [len(df_population_processed)],
        "columns": [df_population_processed.shape[1]],
}))

Für eine spätere Analyse kann es sinnvoll sein das Alter als einen numerischen Typ zu haben

In [ ]:
df_population_processed.rename(columns={
    "Alter": "alter_label",
}, inplace=True)
df_population_processed["alter"] = df_population_processed["alter_label"].str.extract(r"(\d+)").astype("float")
df_population_processed.head()

Für eingacheres handling macht es Sinn die Spalten umzubennen.

In [ ]:
df_population_processed = df_population_processed.rename(columns={
    "Jahr": "year",
    "Kanton": "canton",
    "alter_label": "age_label",
    "Bestand am 31. Dezember": "population",
    "alter": "age",
})
df_population_processed.head()

In [ ]:
pd.DataFrame({
        "year_min": [df_population_processed["year"].min()],
        "year_max": [df_population_processed["year"].max()],
        "n_years": [df_population_processed["year"].nunique()],
        "n_cantons": [df_population_processed["canton"].nunique()],
        "n_age_groups": [df_population_processed["age_label"].nunique()],
})

In [ ]:
df_population_ch = df_population_processed[
    (df_population_processed["canton"] == "Schweiz") &
    (df_population_processed["age_label"].eq("Alter - Total"))
].copy()

px.line(
    df_population_ch,
    x="year",
    y="population",
    markers=True,
    title="Bevölkerung Schweiz, Bestand am 31. Dezember",
    labels={"year": "Jahr", "population": "Bestand am 31. Dezember"},
)

In [ ]:
df_ch = df_population_processed[df_population_processed["canton"] == "Schweiz"]

official_total = df_ch[df_ch["age_label"] == "Alter - Total"].set_index("year")["population"]

age_sum = (
    df_ch[df_ch["age_label"] != "Alter - Total"]
    .groupby("year")["population"]
    .sum()
)

(age_sum - official_total).abs().max()

In [ ]:
df_population_age = df_population_processed[df_population_processed["age_label"] != "Alter - Total"].copy()
df_population_age["is_66_plus"] = df_population_age["age"] >= 66

df_population_66_share = (
    df_population_age
    .groupby(["year", "canton", "is_66_plus"], as_index=False)["population"]
    .sum()
)

df_population_66_share["total_population"] = df_population_66_share.groupby(["year", "canton"])["population"].transform("sum")
df_population_66_share["population_share_66_plus"] = df_population_66_share["population"] / df_population_66_share["total_population"]
df_population_66_share = df_population_66_share[df_population_66_share["is_66_plus"]].copy()

df_population_66_share.head()

In [ ]:
px.line(
    df_population_66_share[df_population_66_share["canton"] == "Schweiz"],
    x="year",
    y="population_share_66_plus",
    markers=True,
    title="Anteil 66+ an der Schweizer Bevölkerung",
    labels={"year": "Jahr", "population_share_66_plus": "Anteil 66+"},
)

In [ ]:
latest_population_year = df_population_66_share["year"].max()

df_population_66_latest = (
    df_population_66_share[
        (df_population_66_share["year"] == latest_population_year) &
        (~df_population_66_share["canton"].isin(["Schweiz"]))
    ]
    .sort_values("population_share_66_plus", ascending=False)
)

px.bar(
    df_population_66_latest.head(10),
    x="canton",
    y="population_share_66_plus",
    title=f"Top 10 Kantone nach 66+-Anteil {latest_population_year}",
    labels={"canton": "Kanton", "population_share_66_plus": "Anteil 66+"},
)

In [ ]:
df_population_processed.to_csv(PROCESSED_DIR / "bevoelkerung.csv", index=False)
df_population_66_share.to_csv(PROCESSED_DIR / "bevoelkerung_66_share.csv", index=False)

## Prämien

Die Prämien liegen in drei historischen Schemas vor. Für die EDA wird eine gemeinsame Minimalansicht erstellt.

In [ ]:
PREMIUM_FILES = {
    "2011_2014": INTERIM_DIR / "premium_2011_2014.csv",
    "2015_2016": INTERIM_DIR / "premium_2015_2016.csv",
    "2017_2026": INTERIM_DIR / "premium_2017_2026.csv",
}
df_11_14 = pd.read_csv(PREMIUM_FILES["2011_2014"])
df_15_16 = pd.read_csv(PREMIUM_FILES["2015_2016"])
df_17_26 = pd.read_csv(PREMIUM_FILES["2017_2026"])

pd.concat([
        pd.DataFrame({
            "rows": [len(df_11_14)],
            "columns": [df_11_14.shape[1]],
            "duplicate_rows": [df_11_14.duplicated().sum()],
            "missing_cells": [df_11_14.isna().sum().sum()],
        }),
        pd.DataFrame({
            "rows": [len(df_15_16)],
            "columns": [df_15_16.shape[1]],
            "duplicate_rows": [df_15_16.duplicated().sum()],
            "missing_cells": [df_15_16.isna().sum().sum()],
        }),
        pd.DataFrame({
            "rows": [len(df_17_26)],
            "columns": [df_17_26.shape[1]],
            "duplicate_rows": [df_17_26.duplicated().sum()],
            "missing_cells": [df_17_26.isna().sum().sum()],
        })
    ]
)

In [ ]:
def missing_by_column(df):
    return (
        df.isna()
        .sum()
        .sort_values(ascending=False)
        .to_frame("missing_count")
    )

for name, df in {
    "2011_2014": df_11_14,
    "2015_2016": df_15_16,
    "2017_2026": df_17_26,
}.items():
    print(name)
    display(missing_by_column(df).head(10))

In [ ]:
df_11_14[df_11_14["isbase_p"].isna()].head()

In [ ]:
df_11_14[df_11_14["isbase_p"].isna()]["jahr_import"].value_counts()


Bei den Fehlenden Werten handelt es sich um solche welche für die Spätere Analyse benötigt werden. Das es aber nur sehr wenige sind können diese problemlos entfern werden.

In [ ]:
premium_2011_2014_columns = [
    "jahr_import",
    "c_id",
    "v2_typ",
    "isbase_p",
    "isbase_f",
    "f",
    "p",
]

df_11_14_missing_required = df_11_14[
    df_11_14[premium_2011_2014_columns].isna().any(axis=1)
].copy()

df_11_14_missing_required["jahr_import"].value_counts().sort_index()

In [ ]:
df_11_14_processed = df_11_14.dropna(
    subset=premium_2011_2014_columns
).copy()

print(f"Verbleibende Zeilen: {len(df_11_14_processed)}")

In [ ]:
df_11_14_processed = pd.DataFrame({
            "year": pd.to_numeric(df_11_14_processed["jahr_import"], errors="coerce"),
            "canton_code": df_11_14_processed["c_id"],
            "age_class": df_11_14_processed["v2_typ"],
            "premium": pd.to_numeric(df_11_14_processed["p"], errors="coerce"),
            "franchise": pd.to_numeric(df_11_14_processed["f"], errors="coerce"),
            "is_base_p": pd.to_numeric(df_11_14_processed["isbase_p"], errors="coerce"),
            "is_base_f": pd.to_numeric(df_11_14_processed["isbase_f"], errors="coerce"),
            "source_schema": "2011-2014",
        })

df_11_14_processed.head()

In [ ]:
df_15_16_processed = pd.DataFrame({
        "year": pd.to_numeric(df_15_16["geschaeftsjahr"], errors="coerce"),
        "canton_code": df_15_16["kanton"],
        "age_class": df_15_16["altersklasse"],
        "premium": pd.to_numeric(df_15_16["praemie"], errors="coerce"),
        "franchise": df_15_16["franchise"],
        "is_base_p": pd.to_numeric(df_15_16["isbasep"], errors="coerce"),
        "is_base_f": pd.to_numeric(df_15_16["isbasef"], errors="coerce"),
        "source_schema": "2015-2016",
    })
df_15_16_processed.head()

In [ ]:
df_17_26_processed = pd.DataFrame({
        "year": pd.to_numeric(df_17_26["geschaeftsjahr"], errors="coerce"),
        "canton_code": df_17_26["kanton"],
        "age_class": df_17_26["altersklasse"],
        "premium": pd.to_numeric(df_17_26["praemie"], errors="coerce"),
        "franchise": df_17_26["franchise"],
        "is_base_p": pd.to_numeric(df_17_26["isbasep"], errors="coerce"),
        "is_base_f": pd.to_numeric(df_17_26["isbasef"], errors="coerce"),
        "source_schema": "2017-2026",
    })
df_17_26_processed.head()

In [ ]:
df_premiums = pd.concat([
    df_11_14_processed,
    df_15_16_processed,
    df_17_26_processed,
])
df_premiums.head()

In [ ]:
pd.DataFrame({
    "year_min": [df_premiums["year"].min()],
    "year_max": [df_premiums["year"].max()],
    "n_years": [df_premiums["year"].nunique()],
    "n_cantons": [df_premiums["canton_code"].nunique()],
    "n_age_classes": [df_premiums["age_class"].nunique()],
    "missing_premium": [df_premiums["premium"].isna().mean()],
})

Wir vergleichen nur die Standardprämien und Franchise

In [ ]:
df_premiums_base = df_premiums[
    (df_premiums["is_base_p"] == 1) &
    (df_premiums["is_base_f"] == 1)]

premium_year = (
    df_premiums_base
    .groupby("year", as_index=False)["premium"]
    .mean()
)

px.line(
    premium_year,
    x="year",
    y="premium",
    markers=True,
    title="Durchschnittliche Basisprämie nach Jahr",
    labels={"year": "Jahr", "premium": "Monatsprämie in CHF"},
)

In [ ]:
df_premiums.to_csv(PROCESSED_DIR / "praemien.csv", index=False)
df_premiums_base.to_csv(PROCESSED_DIR / "praemien_base.csv", index=False)